In [1]:
# pip install mygene

In [47]:
import pandas as pd
import numpy as np
import requests
import time
import mygene

In [48]:
df = pd.read_csv("datosGene4PD/t_common_variant.txt", sep = "\t", index_col = False)

In [49]:
df

,Chr,gene_symbol,SNPs_symbol,SNP_position,effect_allele,alternate_allele,joint_phase_P,joint_phase_OR,joint_phase_OR_CI,pubMed_ID,Unnamed: 10
0,6,GPR126,rs757765789,142758601,T,G,1.42E-06,1.06,1.016-1.098,28256260,NaN
1,1,SYT11,rs202015799,155839054,C,T,4.70E-09,-,-,24842889,NaN
2,12,SLC2A13,rs1994090,40428561,G,T,3.20E-54,12.05,8.35-17.41,24842889,NaN
3,12,SLC2A13,rs2708453,40478652,G,T,3.62E-54,12.05,8.35-17.41,24842889,NaN
4,12,SLC2A13,rs4768212,40474147,C,T,3.62E-54,12.05,8.35-17.41,24842889,NaN
...,...,...,...,...,...,...,...,...,...,...,...
1053,rs117896735,INPP5F,13,U,13,U,1.21E-11,1.77,-,29700661,NaN
1054,rs12456492,RIT2,21,U,21,U,2.15E-11,1.1,-,29700661,NaN
1055,rs7155501,GCH1,15,U,15,U,1.25E-10,1.12,-,29700661,NaN
1056,rs10797576,SIPA1L2,21,U,21,U,1.76E-10,1.13,-,29700661,NaN


In [52]:
def extrae_gene_symbols(dataframe):
    """
    Función destinada a limpiar, acondicionar y extraer la columna 'gene_symbol' a partir del DataFrame original. Elimina valores <na> de
    dicha columna, elimina posibles gene symbols que incluyan el término 'dist' que complica las búsquedas posteriores, y separa en líneas
    diferentes los gene symbols que incluyen varios genes separados por ',' o ';'.

    Parámetros
    -------------------------------
    - dataframe (pandas.core.frame.DataFrame) -> DataFrame original importado desde t_common_variant.txt

    Returns
    -------------------------------
    - df_corregido (pandas.core.frame.DataFrame) -> DataFrame original corregido tras eliminar columna espuria al final, y valores <na> 
                                                    de la columna 'gene_symbol'
    - lista_symbols (list) -> lista con todos los gene symbols de la columna 'gene_symbols', tras separar los genes que aparecen 
                              originalmente unidos por ',' o ';', y eliminar las entradas con 'dist' y valores <na>
    - lista_unicos (list) -> lista todos los valores diferentes presentes en lista_symbols, sin repetidos.
    """

    df_corregido = df.drop('Unnamed: 10', axis = 1)
    df_corregido = df_corregido.dropna(subset = ['gene_symbol'])
    df_corregido = df_corregido.reset_index(drop=False)
    df_corregido = df_corregido.drop('index', axis = 1)
    df_genes = df_corregido["gene_symbol"]
    
    lista_symbols = []
    
    for i, gene in enumerate(df_genes):
        
        gene = gene.replace(",", ";")
    
        if "dist" in gene:
            continue

        elif ";" in gene:

            separacion1 = gene.split(";")

            for gen in separacion1:
                lista_symbols.append(gen)

        else:
            lista_symbols.append(gene)

    lista_unicos = []
    
    for symbol in lista_symbols:
        
        if symbol not in lista_unicos:
            
            lista_unicos.append(symbol)
                
    return df_corregido, lista_symbols, lista_unicos

In [53]:
df_corregido, lista_symbols, lista_unicos = extrae_gene_symbols(df)

In [54]:
df_corregido

,Chr,gene_symbol,SNPs_symbol,SNP_position,effect_allele,alternate_allele,joint_phase_P,joint_phase_OR,joint_phase_OR_CI,pubMed_ID
0,6,GPR126,rs757765789,142758601,T,G,1.42E-06,1.06,1.016-1.098,28256260
1,1,SYT11,rs202015799,155839054,C,T,4.70E-09,-,-,24842889
2,12,SLC2A13,rs1994090,40428561,G,T,3.20E-54,12.05,8.35-17.41,24842889
3,12,SLC2A13,rs2708453,40478652,G,T,3.62E-54,12.05,8.35-17.41,24842889
4,12,SLC2A13,rs4768212,40474147,C,T,3.62E-54,12.05,8.35-17.41,24842889
...,...,...,...,...,...,...,...,...,...,...
1051,rs117896735,INPP5F,13,U,13,U,1.21E-11,1.77,-,29700661
1052,rs12456492,RIT2,21,U,21,U,2.15E-11,1.1,-,29700661
1053,rs7155501,GCH1,15,U,15,U,1.25E-10,1.12,-,29700661
1054,rs10797576,SIPA1L2,21,U,21,U,1.76E-10,1.13,-,29700661


In [56]:
def extrae_SNPs(df_corregido):
    """
    Función destinada a adaptar df_corregido para que contenga únicamente las columnas de interés para el manejo de los SNPs ('Chr', 
    'gene_symbol', 'SNP_position', 'effect_allele', 'alternate_allele'). Se separan en líneas diferentes las entradas correspondientes 
    a los genes que se presentan unidos por ',' o ';' en df_corregido. Se eliminan las entradas espurias que, por motivos ajenos, ya sea
    por el funcionamiento del procesador de texto o el estado del dataset original, presentan datos corruptos, como el SNP_symbol en la 
    columna de cromosomas 'Chr'.

    Parámetros
    -------------------------------
    - df_corregido (pandas.core.frame.DataFrame) -> DataFrame original corregido tras eliminar columna espuria al final, y valores <na> 
                                                    de la columna 'gene_symbol'
    Returns
    -------------------------------
    - df_SNPs (pandas.core.frame.DataFrame) -> DataFrame con las columnas ('Chr', 'gene_symbol', 'SNP_position', 'effect_allele', 
    'alternate_allele') limpias y acondicionadas.
    """
    df_corregido["gene_symbol"] = df_corregido["gene_symbol"].str.split(r'\s*[;,]\s*')

    df_corregido = df_corregido.explode("gene_symbol").reset_index(drop = True)

    df_SNPs = df_corregido[["Chr", "gene_symbol", "SNP_position", "effect_allele", "alternate_allele"]]

    df_SNPs = df_SNPs[~df_SNPs['Chr'].str.contains('rs', na = False)]

    df_SNPs = df_SNPs[~df_SNPs['gene_symbol'].str.contains('dist')].reset_index(drop = True)

    df_SNPs = df_SNPs[df_SNPs['gene_symbol'] != 'NONE'].reset_index(drop = True)

    df_SNPs = df_SNPs[df_SNPs['Chr'] != '-'].reset_index(drop = True)
    
    return df_SNPs

In [57]:
df_SNPs = extrae_SNPs(df_corregido)

In [58]:
df_SNPs

,Chr,gene_symbol,SNP_position,effect_allele,alternate_allele
0,6,GPR126,142758601,T,G
1,1,SYT11,155839054,C,T
2,12,SLC2A13,40428561,G,T
3,12,SLC2A13,40478652,G,T
4,12,SLC2A13,40474147,C,T
...,...,...,...,...,...
1344,17,DNAH17,76425480,A,T
1345,18,ASXL3,31304318,T,G
1346,18,MEX3C,48683589,T,G
1347,20,CRLS1,6006041,T,C


In [60]:
lista_genes_snps = []
lista_unicos_snps = []

for i in range(len(df_SNPs)):
    if df_SNPs.iloc[i]["gene_symbol"] not in lista_genes_snps:
        lista_genes_snps.append(df_SNPs.iloc[i]["gene_symbol"])
        lista_unicos_snps.append(df_SNPs.iloc[i]["gene_symbol"])
    else:
        lista_genes_snps.append(df_SNPs.iloc[i]["gene_symbol"])       

In [61]:
conteo_chr = df_SNPs.groupby("gene_symbol")["Chr"].nunique()
genes_problema = conteo_chr[conteo_chr > 1].index
df_SNPs = df_SNPs[~df_SNPs["gene_symbol"].isin(genes_problema)]
df_SNPs = df_SNPs.reset_index(drop = True)

In [62]:
df_SNPs

,Chr,gene_symbol,SNP_position,effect_allele,alternate_allele
0,6,GPR126,142758601,T,G
1,1,SYT11,155839054,C,T
2,12,SLC2A13,40428561,G,T
3,12,SLC2A13,40478652,G,T
4,12,SLC2A13,40474147,C,T
...,...,...,...,...,...
1282,17,DNAH17,76425480,A,T
1283,18,ASXL3,31304318,T,G
1284,18,MEX3C,48683589,T,G
1285,20,CRLS1,6006041,T,C


In [64]:
def extrae_coords_inicio_fin(df_SNPs, lista_unicos_snps):

    chr_esperado = dict(zip(df_SNPs["gene_symbol"], df_SNPs["Chr"].astype(str)))
    
    mg = mygene.MyGeneInfo()
    
    resultados_coords = mg.querymany(lista_unicos_snps, scopes = "symbol,alias", fields = "genomic_pos_hg19", species = "human")
    
    coords_genes = []
    no_encontrados = []
    genes_procesados = set()
    
    for resultado in resultados_coords:

        gen = resultado.get("query")
        
        if resultado.get("notfound"):
            if gen not in no_encontrados:
                no_encontrados.append(gen)
            continue

    recuperados = [gen.replace('LOC','') for gen in no_encontrados if 'LOC' in gen]

    recuperados_coords_aux = mg.querymany(recuperados, scopes = "entrezgene", fields = "symbol,genomic_pos_hg19", species = "human")
    
    nuevos_symbols = [recuperados_coords_aux[i]["symbol"] for i, gen in enumerate(recuperados_coords_aux) if not gen.get("notfound")]

    recuperados_coords = mg.querymany(nuevos_symbols, scopes = "symbol,alias", fields = "genomic_pos_hg19", species = "human")

    for resultado in resultados_coords:

        gen = resultado.get("query")
        
        if gen in genes_procesados:
            continue
    
        posiciones = resultado.get("genomic_pos_hg19")
        
        if not posiciones:
            continue
    
        if not isinstance(posiciones, list):
            posiciones = [posiciones]
    
        chr_buscado = chr_esperado.get(gen)
        posicion_elegida = None
    
        for pos in posiciones:
            if str(pos.get("chr")) == chr_buscado:
                posicion_elegida = pos
                break
    
        if posicion_elegida:
            coords_genes.append({"gene_symbol": gen, "chr": str(posicion_elegida.get("chr")), "inicio": posicion_elegida.get("start"), "fin": posicion_elegida.get("end"), "cadena": posicion_elegida.get("strand")})
    
            genes_procesados.add(gen)
    
        if gen in no_encontrados:
            no_encontrados.remove(gen)

    for resultado in recuperados_coords:

        gen = "LOC" + resultado.get("_id")
        
        if gen in genes_procesados:
            continue
    
        posiciones = resultado.get("genomic_pos_hg19")
        
        if not posiciones:
            continue
    
        if not isinstance(posiciones, list):
            posiciones = [posiciones]
    
        chr_buscado = chr_esperado.get(gen)
        posicion_elegida = None
    
        for pos in posiciones:
            if str(pos.get("chr")) == chr_buscado:
                posicion_elegida = pos
                break
    
        if posicion_elegida:
            coords_genes.append({"gene_symbol": gen, "chr": str(posicion_elegida.get("chr")), "inicio": posicion_elegida.get("start"), "fin": posicion_elegida.get("end"), "cadena": posicion_elegida.get("strand")})
    
            genes_procesados.add(gen)
    
        if gen in no_encontrados:
            no_encontrados.remove(gen)

    df_coords = pd.DataFrame(coords_genes)

    return df_coords, no_encontrados

In [65]:
df_coords, no_encontrados = extrae_coords_inicio_fin(df_SNPs, lista_unicos_snps)

43 input query terms found dup hits:	[('TBC1D3P2', 2), ('CAST', 4), ('DHFRP3', 2), ('HLA-DQB1', 2), ('BST1', 2), ('CASC6', 2), ('PWRN4', 
47 input query terms found no hit:	['SLC2A15', 'LOC440311', '43160', 'LOC100133091', 'LOC101928978', 'MIR7641-2', 'LOC201175', 'LOC1019
7 input query terms found no hit:	['201175', '100129900', '100129831', '645177', '100130911', '729160', '100132423']
12 input query terms found dup hits:	[('BALR6', 2), ('COX6CP4', 2), ('CLIC4P1', 2), ('CRIM1-DT', 2), ('SELENOKP3', 2), ('RPL7L1P5', 2), (


In [66]:
df_coords

,gene_symbol,chr,inicio,fin,cadena
0,GPR126,6,142622991,142767403,1
1,SYT11,1,155829300,155854990,1
2,SLC2A13,12,40148823,40499891,-1
3,LRRK2,12,40590546,40763087,1
4,GPRIN3,4,90157537,90229161,-1
...,...,...,...,...,...
578,LOC100129138,1,104615645,104619709,1
579,LOC101929066,8,17942377,17953903,1
580,LOC100507657,22,27706612,27713417,1
581,LOC284930,22,48027423,48251349,1


In [68]:
df_completo = pd.merge(df_SNPs, df_coords, on = "gene_symbol")

In [69]:
df_completo

,Chr,gene_symbol,SNP_position,effect_allele,alternate_allele,chr,inicio,fin,cadena
0,6,GPR126,142758601,T,G,6,142622991,142767403,1
1,1,SYT11,155839054,C,T,1,155829300,155854990,1
2,1,SYT11,154105678,T,C,1,155829300,155854990,1
3,1,SYT11,154105678,T,C,1,155829300,155854990,1
4,1,SYT11,155359992,T,C,1,155829300,155854990,1
...,...,...,...,...,...,...,...,...,...
1156,17,DNAH17,76425480,A,T,17,76419778,76573476,-1
1157,18,ASXL3,31304318,T,G,18,31158579,31331156,1
1158,18,MEX3C,48683589,T,G,18,48700920,48744674,-1
1159,20,CRLS1,6006041,T,C,20,5986736,6020699,1


In [71]:
df_completo = df_completo.drop_duplicates().reset_index(drop=True)

In [72]:
df_completo

,Chr,gene_symbol,SNP_position,effect_allele,alternate_allele,chr,inicio,fin,cadena
0,6,GPR126,142758601,T,G,6,142622991,142767403,1
1,1,SYT11,155839054,C,T,1,155829300,155854990,1
2,1,SYT11,154105678,T,C,1,155829300,155854990,1
3,1,SYT11,155359992,T,C,1,155829300,155854990,1
4,12,SLC2A13,40428561,G,T,12,40148823,40499891,-1
...,...,...,...,...,...,...,...,...,...
1043,17,DNAH17,76425480,A,T,17,76419778,76573476,-1
1044,18,ASXL3,31304318,T,G,18,31158579,31331156,1
1045,18,MEX3C,48683589,T,G,18,48700920,48744674,-1
1046,20,CRLS1,6006041,T,C,20,5986736,6020699,1


In [73]:
df_completo_limpio = df_completo[df_completo["Chr"] == df_completo["chr"]].reset_index(drop = True)

df_completo_limpio = df_completo_limpio[~df_completo_limpio['SNP_position'].str.contains('-', na = False)].copy()

df_completo_limpio = df_completo_limpio.reset_index(drop = True)

In [74]:
df_completo_limpio

,Chr,gene_symbol,SNP_position,effect_allele,alternate_allele,chr,inicio,fin,cadena
0,6,GPR126,142758601,T,G,6,142622991,142767403,1
1,1,SYT11,155839054,C,T,1,155829300,155854990,1
2,1,SYT11,154105678,T,C,1,155829300,155854990,1
3,1,SYT11,155359992,T,C,1,155829300,155854990,1
4,12,SLC2A13,40428561,G,T,12,40148823,40499891,-1
...,...,...,...,...,...,...,...,...,...
1042,17,DNAH17,76425480,A,T,17,76419778,76573476,-1
1043,18,ASXL3,31304318,T,G,18,31158579,31331156,1
1044,18,MEX3C,48683589,T,G,18,48700920,48744674,-1
1045,20,CRLS1,6006041,T,C,20,5986736,6020699,1


In [75]:
df_completo_limpio["SNP_position"] = df_completo_limpio["SNP_position"].astype(int)

In [76]:
df_completo_limpio.dtypes

Chr                 object
gene_symbol         object
SNP_position         int32
effect_allele       object
alternate_allele    object
chr                 object
inicio               int64
fin                  int64
cadena               int64
dtype: object

In [77]:
df_final = df_completo_limpio[(df_completo_limpio["SNP_position"] >= df_completo_limpio["inicio"]) & (df_completo_limpio["SNP_position"] <= df_completo_limpio["fin"])].copy()

In [78]:
df_final = df_final.reset_index(drop = True)

In [79]:
df_final

,Chr,gene_symbol,SNP_position,effect_allele,alternate_allele,chr,inicio,fin,cadena
0,6,GPR126,142758601,T,G,6,142622991,142767403,1
1,1,SYT11,155839054,C,T,1,155829300,155854990,1
2,12,SLC2A13,40428561,G,T,12,40148823,40499891,-1
3,12,SLC2A13,40478652,G,T,12,40148823,40499891,-1
4,12,SLC2A13,40474147,C,T,12,40148823,40499891,-1
...,...,...,...,...,...,...,...,...,...
532,17,BRIP1,59917366,T,C,17,59758627,59940882,-1
533,17,DNAH17,76425480,A,T,17,76419778,76573476,-1
534,18,ASXL3,31304318,T,G,18,31158579,31331156,1
535,20,CRLS1,6006041,T,C,20,5986736,6020699,1


In [80]:
df_final["inicio"] = df_final["inicio"].astype(int)
df_final["fin"] = df_final["fin"].astype(int)

In [81]:
df_final = df_final.drop('chr', axis = 1)

In [82]:
df_final = df_final[['Chr', 'gene_symbol', 'inicio', 'SNP_position', 'fin', 'effect_allele', 'alternate_allele', 'cadena']]

In [83]:
df_final

,Chr,gene_symbol,inicio,SNP_position,fin,effect_allele,alternate_allele,cadena
0,6,GPR126,142622991,142758601,142767403,T,G,1
1,1,SYT11,155829300,155839054,155854990,C,T,1
2,12,SLC2A13,40148823,40428561,40499891,G,T,-1
3,12,SLC2A13,40148823,40478652,40499891,G,T,-1
4,12,SLC2A13,40148823,40474147,40499891,C,T,-1
...,...,...,...,...,...,...,...,...
532,17,BRIP1,59758627,59917366,59940882,T,C,-1
533,17,DNAH17,76419778,76425480,76573476,A,T,-1
534,18,ASXL3,31158579,31304318,31331156,T,G,1
535,20,CRLS1,5986736,6006041,6020699,T,C,1


In [84]:
df_final.to_csv('datosGene4PD/coordenadas_genes_nuevo.csv', index = False)